# Fixed points of 2-qubit distillation maps for different code choices

For each choice of the 2-qubit stabilizer $S$ and logical operators $\bar X$, $\bar Z$ (cell headings), the cell writes down the accepted-branch Bloch map $(x,y,z)\mapsto(x',y',z')$ by hand. `analyze_map_numerical` then searches a 9×9×9 grid of starting points with `scipy.optimize.root` for fixed points on the Bloch sphere ($|r|=1$) and prints the moduli of the Jacobian eigenvalues there. A fixed point with some eigenvalue moduli below 1 attracts nearby states in those directions.

**Not part of the data or figure pipeline**; it supports the choice of code in the text.

In [37]:
import sympy as sp
import numpy as np

from scipy.optimize import root

# Global function 

In [38]:
def analyze_map_numerical(mapping, variables,
                          bounds=(-1, 1),
                          grid_size=7,
                          tol=1e-8,
                          norm_tol=1e-6):

    variables = sp.Matrix(variables)
    mapping = sp.Matrix(mapping)

    F = mapping - variables
    J = mapping.jacobian(variables)
    symbols = tuple(variables)

    f_numeric = sp.lambdify(
        symbols, F, "numpy"
    )

    j_numeric = sp.lambdify(
        symbols, J, "numpy"
    )

    grid = np.linspace(bounds[0], bounds[1], grid_size)

    fixed_points = []

    for x0 in grid:
        for y0 in grid:
            for z0 in grid:

                initial = np.array([x0, y0, z0])

                try:
                    sol = root(
                        lambda p: np.asarray(
                            f_numeric(*p),
                            dtype=float
                        ).flatten(),
                        initial
                    )

                    p = sol.x

                    residual = np.asarray(
                        f_numeric(*p),
                        dtype=float
                    ).flatten()

                    if (
                        np.all(np.isfinite(p))
                        and np.all(np.isfinite(residual))
                        and np.linalg.norm(residual) < tol
                        and abs(np.linalg.norm(p) - 1.0) < norm_tol
                        and not any(
                            np.linalg.norm(p - q) < 1e-6
                            for q in fixed_points
                        )
                    ):
                        fixed_points.append(p)

                except (ValueError, OverflowError,
                        ZeroDivisionError, FloatingPointError):
                    continue

    results = []

    for p in fixed_points:

        Jp = np.asarray(
            j_numeric(*p), dtype=float
        )
        eigs = np.linalg.eigvals(Jp)

        results.append({
            "point": p,
            "jacobian": Jp,
            "eigenvalues": eigs
        })

    return results

SyntaxError: '(' was never closed (850215113.py, line 68)

# $S = ZZ$ , $\bar{X} = ZI$ , $\bar{Z} = XX$

In [ ]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    2*z / denominator,
    -2*x*y / denominator,
    (x**2 - y**2) / denominator,
])


results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-0.54368901 -0.7855089  -0.29559774]
|r| =  1.0000000002465022
|Eigenvalues|: [1.75709363 1.75709363 1.54368901]

Fixed point: [-0.54368901  0.7855089  -0.29559774]
|r| =  1.0000000000000069
|Eigenvalues|: [1.75709363 1.75709363 1.54368901]

Fixed point: [ 8.39286755e-01 -1.24656368e-11  5.43689012e-01]
|r| =  0.9999999998212749
|Eigenvalues|: [0.83928676 1.29559774 1.29559774]



# $S = ZZ$ , $\bar{X} = XX$ , $\bar{Z} = IZ$ 

In [26]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    (x**2 - y**2) / denominator,
    2*x*y / denominator,
    2*z / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [ 7.05008250e-16 -5.40801253e-17 -1.00000000e+00]
|r| =  1.0
|Eigenvalues|: [7.07079411e-16 7.07079411e-16 0.00000000e+00]

Fixed point: [ 1.79373339e-14 -9.29365939e-16  1.00000000e+00]
|r| =  1.0000000000000064
|Eigenvalues|: [1.79613938e-14 1.79613938e-14 6.43929354e-15]

Fixed point: [1.00000000e+00 1.64932237e-11 1.27406868e-14]
|r| =  0.9999999999927438
|Eigenvalues|: [2. 2. 2.]



# $S = ZZ$ , $\bar{X} = IZ$ , $\bar{Z} = XY$

In [27]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    2*z / denominator,
    (x**2 - y**2) / denominator,
    2*x*y / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-1.70050249e-13 -1.00000000e+00 -1.20703492e-13]
|r| =  1.000000000000123
|Eigenvalues|: [2. 2. 2.]

Fixed point: [-0.78615137  0.38196601 -0.48586827]
|r| =  0.9999999938251768
|Eigenvalues|: [1.00000001 1.41421356 1.41421356]

Fixed point: [0.78615137 0.38196601 0.48586827]
|r| =  0.9999999947940708
|Eigenvalues|: [1.00000001 1.41421356 1.41421356]



# $S = ZZ$ , $\bar{X} = XY$ , $\bar{Z} = IZ$ 

In [28]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    2*x*y / denominator,
    (y**2 - x**2) / denominator,
    2*z / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-5.40801253e-17  7.05008251e-16 -1.00000000e+00]
|r| =  1.0
|Eigenvalues|: [7.07079411e-16 7.07079411e-16 0.00000000e+00]

Fixed point: [-9.29371766e-16  1.79373402e-14  1.00000000e+00]
|r| =  1.0000000000000064
|Eigenvalues|: [1.79614004e-14 1.79614004e-14 6.43929354e-15]

Fixed point: [-2.63093586e-12  1.00000000e+00  1.06967328e-12]
|r| =  1.0000000000002438
|Eigenvalues|: [2. 2. 2.]



# $S = ZZ$ , $\bar{X} = XX$ , $\bar{Z} = XY$

In [29]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    (x**2 - y**2) / denominator,
    -2*z / denominator,
    2*x*y / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-0.38196601 -0.78615138  0.48586827]
|r| =  0.9999999979747849
|Eigenvalues|: [1.41421356 1.41421356 1.        ]

Fixed point: [-0.38196601  0.78615138 -0.48586827]
|r| =  0.9999999959390157
|Eigenvalues|: [1.41421356 1.41421356 1.        ]

Fixed point: [ 1.00000000e+00 -3.46763257e-11  1.73381626e-11]
|r| =  1.0000000000000002
|Eigenvalues|: [2. 2. 2.]



# $S = ZZ$ , $\bar{X} = XY$ , $\bar{Z} = XX$ 

In [30]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    2*x*y / denominator,
    2*z / denominator,
    (x**2 - y**2) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [ 4.77364870e-13 -8.39286755e-01 -5.43689013e-01]
|r| =  0.9999999999905869
|Eigenvalues|: [1.29559774 1.29559774 0.83928676]

Fixed point: [-0.7855089   0.54368901  0.29559774]
|r| =  1.0000000000002793
|Eigenvalues|: [1.54368901 1.75709363 1.75709363]

Fixed point: [0.7855089  0.54368901 0.29559774]
|r| =  0.9999999999991153
|Eigenvalues|: [1.54368901 1.75709363 1.75709363]



# $S = ZZ$, $\overline{X} = XX$, $\overline{Z} = -IZ$

In [39]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + z**2

mapping = sp.Matrix([
    (x**2 - y**2) / denominator,
    -2*x*y / denominator,
    -2*z / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-5.00000000e-01 -8.66025404e-01  1.23819168e-14]
|r| =  0.9999999999998893
|Eigenvalues|: [2. 2. 2.]

Fixed point: [-5.00000000e-01  8.66025404e-01  2.35099232e-19]
|r| =  0.9999999999999911
|Eigenvalues|: [2. 2. 2.]

Fixed point: [ 1.00000000e+00 -1.28830116e-12  2.12812083e-17]
|r| =  1.00000000000136
|Eigenvalues|: [2. 2. 2.]



# $S =  XY$, $\bar{X} = IY$, $\bar{Z} = YX$

In [31]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    (x+y) / denominator,
    z*(y - x) / denominator,
    (x*y + z**2) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-1.00000000e+00  8.08330167e-15 -3.49494288e-14]
|r| =  1.0
|Eigenvalues|: [1. 1. 1.]

Fixed point: [ 1.61156234e-12 -6.44678281e-12  1.00000000e+00]
|r| =  0.9999999999932084
|Eigenvalues|: [1.41421356 1.41421356 2.        ]

Fixed point: [ 1.00000000e+00  3.62878930e-17 -3.40422332e-17]
|r| =  0.9999999999999986
|Eigenvalues|: [1. 1. 1.]



<lambdifygenerated-37>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y + z*(-x + y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])
<lambdifygenerated-37>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y + z*(-x + y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])


# $S = XY $, $\bar{X} = YX$, $\bar{Z} = IY$

In [32]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    (x*y + z**2) / denominator,
    z*(x - y) / denominator,
    (x + y) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [ 0.24328132 -0.74819367 -0.61726852]
|r| =  0.9999999999180956
|Eigenvalues|: [1.61496738 1.61496738 2.06403175]

Fixed point: [0.61726852 0.24328132 0.74819367]
|r| =  1.0000000000000309
|Eigenvalues|: [1.17907976 0.97572886 0.97572886]

Fixed point: [-0.74819367  0.61726852 -0.24328132]
|r| =  1.000000000000086
|Eigenvalues|: [3.65715623 2.61656516 2.61656516]



<lambdifygenerated-39>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (x*y + z**2)/(x*y + 1)], [-y + z*(x - y)/(x*y + 1)], [-z + (x + y)/(x*y + 1)]])
<lambdifygenerated-39>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (x*y + z**2)/(x*y + 1)], [-y + z*(x - y)/(x*y + 1)], [-z + (x + y)/(x*y + 1)]])


# $S = XY $, $\bar{X} = IY$, $\bar{Z} = YZ$

In [33]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    (x + y) / denominator,
    -(x*y + z**2) / denominator,
    z*(y - x) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-1.00000000e+00 -8.22083022e-17 -7.49268781e-17]
|r| =  1.0
|Eigenvalues|: [1. 1. 1.]

Fixed point: [-9.99999857e-01 -1.98951125e-06 -1.63507566e-06]
|r| =  0.9999998567748232
|Eigenvalues|: [0.99999618 0.9999958  0.9999958 ]

Fixed point: [-9.99999857e-01 -1.98951213e-06  1.63511367e-06]
|r| =  0.9999998567240216
|Eigenvalues|: [0.99999618 0.9999958  0.9999958 ]

Fixed point: [-1.00000045e+00  2.90349151e-06  1.09194652e-06]
|r| =  1.000000453663415
|Eigenvalues|: [1.00000494 1.00000669 1.00000669]

Fixed point: [-9.99999129e-01  2.53742642e-06 -1.52456269e-06]
|r| =  0.9999991289405675
|Eigenvalues|: [1.00000536 1.00000406 1.00000406]

Fixed point: [-1.00000094e+00 -1.12607337e-06  2.98089083e-07]
|r| =  1.0000009377176686
|Eigenvalues|: [0.99999836 0.99999836 0.99999839]

Fixed point: [-1.00000022e+00 -3.07314424e-06  9.31645580e-09]
|r| =  1.0000002247334374
|Eigenvalues|: [0.99999397 0.99999397 0.99999408]

Fixed point: [-1.00000046e+00 -6.31055639e-06  1.78394461e-0

<lambdifygenerated-41>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y + (-x*y - z**2)/(x*y + 1)], [z*(-x + y)/(x*y + 1) - z]])
<lambdifygenerated-41>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y + (-x*y - z**2)/(x*y + 1)], [z*(-x + y)/(x*y + 1) - z]])


# $S = XY $, $\bar{X} = YZ$, $\bar{Z} = IY$

In [34]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    z*(y - x) / denominator,
    (x*y + z**2) / denominator,
    (x + y) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-0.74819367  0.24328132 -0.61726852]
|r| =  0.999999999918096
|Eigenvalues|: [2.06403175 1.61496738 1.61496738]

Fixed point: [ 0.61726852 -0.74819367 -0.24328132]
|r| =  1.000000000147733
|Eigenvalues|: [3.65715623 2.61656516 2.61656516]

Fixed point: [0.24328132 0.61726852 0.74819367]
|r| =  0.9999999994764234
|Eigenvalues|: [1.17907976 0.97572886 0.97572886]



<lambdifygenerated-43>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + z*(-x + y)/(x*y + 1)], [-y + (x*y + z**2)/(x*y + 1)], [-z + (x + y)/(x*y + 1)]])
<lambdifygenerated-43>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + z*(-x + y)/(x*y + 1)], [-y + (x*y + z**2)/(x*y + 1)], [-z + (x + y)/(x*y + 1)]])


# $S =  XY $, $\bar{X} = YX$, $\bar{Z} = YZ$

In [35]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    (x*y + z**2) / denominator,
    (x + y) / denominator,
    z*(y - x) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-1.43705620e-13 -1.00000000e+00 -1.03820026e-13]
|r| =  1.0
|Eigenvalues|: [1. 1. 1.]

Fixed point: [6.40712133e-11 1.00000000e+00 3.51154352e-12]
|r| =  1.000000000062498
|Eigenvalues|: [1. 1. 1.]

Fixed point: [-1.35982235e-06  9.99999406e-01 -7.06644846e-07]
|r| =  0.9999994055587295
|Eigenvalues|: [1.0000021  1.0000021  1.00000278]

Fixed point: [-2.53592707e-06  9.99999126e-01 -1.51814577e-06]
|r| =  0.9999991257299727
|Eigenvalues|: [1.00000406 1.00000406 1.00000535]

Fixed point: [ 1.98876410e-06  9.99999854e-01 -1.63652748e-06]
|r| =  0.9999998544679511
|Eigenvalues|: [0.9999958  0.9999958  0.99999618]

Fixed point: [-2.91083762e-06  1.00000048e+00  1.03665934e-06]
|r| =  1.0000004767184378
|Eigenvalues|: [1.00000677 1.00000677 1.00000487]

Fixed point: [1.48007197e-06 1.00000045e+00 4.32360081e-07]
|r| =  1.000000448908638
|Eigenvalues|: [0.99999743 0.99999743 0.99999716]

Fixed point: [ 6.26231380e-06  9.99999267e-01 -1.38022748e-06]
|r| =  0.9999992668158967
|E

<lambdifygenerated-45>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (x*y + z**2)/(x*y + 1)], [-y + (x + y)/(x*y + 1)], [z*(-x + y)/(x*y + 1) - z]])
<lambdifygenerated-45>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (x*y + z**2)/(x*y + 1)], [-y + (x + y)/(x*y + 1)], [z*(-x + y)/(x*y + 1) - z]])


# $S =  XY $, $\bar{X} = YZ$, $\bar{Z} = YX$

In [36]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    z*(y - x) / denominator,
    -(x + y) / denominator,
    (x*y + z**2) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [3.52656206e-14 3.71292193e-15 1.00000000e+00]
|r| =  1.0
|Eigenvalues|: [1.41421356 1.41421356 2.        ]

Fixed point: [ 0.78615138 -0.48586827 -0.38196601]
|r| =  0.9999999999660942
|Eigenvalues|: [3.         2.23606798 2.23606798]

Fixed point: [-0.78615138  0.48586827 -0.38196601]
|r| =  0.9999999999965188
|Eigenvalues|: [3.         2.23606798 2.23606798]



<lambdifygenerated-47>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + z*(-x + y)/(x*y + 1)], [-y + (-x - y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])
<lambdifygenerated-47>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + z*(-x + y)/(x*y + 1)], [-y + (-x - y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])


# $S = XY$, $\overline{X} = -IY$, $\overline{Z} = YX$

In [40]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    -(x + y) / denominator,
    -z*(y - x) / denominator,
    (x*y + z**2) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [3.71292193e-15 3.52656206e-14 1.00000000e+00]
|r| =  1.0
|Eigenvalues|: [1.41421356 1.41421356 2.        ]

Fixed point: [-0.48586827  0.78615138 -0.38196601]
|r| =  0.999999999999967
|Eigenvalues|: [3.         2.23606798 2.23606798]

Fixed point: [ 0.48586827 -0.78615138 -0.38196601]
|r| =  1.0000000001232372
|Eigenvalues|: [3.         2.23606798 2.23606798]



<lambdifygenerated-51>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (-x - y)/(x*y + 1)], [-y - z*(-x + y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])
<lambdifygenerated-51>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (-x - y)/(x*y + 1)], [-y - z*(-x + y)/(x*y + 1)], [-z + (x*y + z**2)/(x*y + 1)]])


# $S = XY$, $\overline{X} = IY$, $\overline{Z} = YX$

In [41]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    (x + y) / denominator,
    -z*(y - x) / denominator,
    -(x*y + z**2) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [-1.00000000e+00 -3.51009309e-14  3.74110293e-13]
|r| =  1.0
|Eigenvalues|: [1. 1. 1.]

Fixed point: [ 5.67727520e-13 -1.15920933e-12 -1.00000000e+00]
|r| =  0.9999999999996801
|Eigenvalues|: [1.41421356 1.41421356 2.        ]

Fixed point: [1.00000000e+00 3.62878930e-17 3.40422332e-17]
|r| =  0.9999999999999986
|Eigenvalues|: [1. 1. 1.]



<lambdifygenerated-53>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y - z*(-x + y)/(x*y + 1)], [-z + (-x*y - z**2)/(x*y + 1)]])
<lambdifygenerated-53>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (x + y)/(x*y + 1)], [-y - z*(-x + y)/(x*y + 1)], [-z + (-x*y - z**2)/(x*y + 1)]])


# $S = XY$, $\overline{X} = -IY$, $\overline{Z} = YZ$

In [42]:
x, y, z = sp.symbols("x y z", real=True)

variables = sp.Matrix([x, y, z])

denominator = 1 + x*y 

mapping = sp.Matrix([
    -(x + y) / denominator,
    -(x*y + z**2) / denominator,
    -z*(y - x) / denominator,
])

results = analyze_map_numerical(
    mapping,
    variables,
    bounds=(-1, 1),
    grid_size=9
)

for result in results:
    print("Fixed point:", result["point"])
    magnitude = 0
    for value in result["point"]:
        magnitude = magnitude + abs(value)**2
    print("|r| = ", np.sqrt(magnitude))
    print("|Eigenvalues|:", abs(result["eigenvalues"]))
    print()

Fixed point: [ 0.29559774 -0.54368901 -0.7855089 ]
|r| =  1.0000000000000009
|Eigenvalues|: [1.6588878  1.6588878  2.29559774]

Fixed point: [ 0.29559774 -0.54368901  0.7855089 ]
|r| =  1.0000000000000009
|Eigenvalues|: [1.6588878  1.6588878  2.29559774]

Fixed point: [-5.43689013e-01  8.39286755e-01  2.49801580e-13]
|r| =  0.9999999999997036
|Eigenvalues|: [3.38297577 2.54368901 2.54368901]



<lambdifygenerated-55>:2: RuntimeWarning: invalid value encountered in scalar divide
  return array([[-x + (-x - y)/(x*y + 1)], [-y + (-x*y - z**2)/(x*y + 1)], [-z*(-x + y)/(x*y + 1) - z]])
<lambdifygenerated-55>:2: RuntimeWarning: divide by zero encountered in scalar divide
  return array([[-x + (-x - y)/(x*y + 1)], [-y + (-x*y - z**2)/(x*y + 1)], [-z*(-x + y)/(x*y + 1) - z]])
